In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


# LOAD DATASET
df = pd.read_csv("student_data.csv")

# REMOVE NULL VALUES
df = df.dropna()

# REMOVE OUTLIERS USING IQR
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if col == 'failures':
        continue

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

# FEATURES AND TARGET
X = df.drop('failures', axis=1)
y = df['failures']

# ONE-HOT ENCODING
X = pd.get_dummies(X, drop_first=True)

# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# FEATURE SCALING
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# CONVERT TARGET TO NUMPY ARRAY
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()


# MINI-BATCH GRADIENT DESCENT CLASSIFIER
class MiniBatchGradientDescentClassifier:
    def __init__(self, learning_rate=0.01, n_iters=1000, batch_size=32):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            # Mini-batches
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i + self.batch_size]
                y_batch = y_shuffled[i:i + self.batch_size]

                batch_size = X_batch.shape[0]

                # Linear model
                linear_output = np.dot(X_batch, self.weights) + self.bias

                # Prediction
                y_predicted = self.sigmoid(linear_output)

                # Gradients
                dw = (1 / batch_size) * np.dot(X_batch.T, (y_predicted - y_batch))
                db = (1 / batch_size) * np.sum(y_predicted - y_batch)

                # Update
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

    def predict_proba(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_output)

    def predict(self, X):
        y_prob = self.predict_proba(X)
        return np.array([1 if i >= 0.5 else 0 for i in y_prob])


# MODEL
model = MiniBatchGradientDescentClassifier(
    learning_rate=0.01,
    n_iters=1000,
    batch_size=32
)

# TRAIN MODEL
model.fit(X_train, y_train)

# PREDICTIONS
y_pred = model.predict(X_test)

# EVALUATION
print("Accuracy Score:")
print(accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy Score:
0.8113207547169812

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.89      0.89        45
           1       0.38      0.60      0.46         5
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1

    accuracy                           0.81        53
   macro avg       0.32      0.37      0.34        53
weighted avg       0.79      0.81      0.80        53



c:\Users\faiza\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\faiza\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\faiza\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave